# Chest X-Ray Classification: CNN + GCN Fusion Model

ResNet-50 feature extractor fused with a Graph Convolutional Network (GCN) for 3-class pneumonia diagnosis (Normal / Bacterial / Viral).

In [ ]:
import subprocess, sys, torch, warnings

# Suppress optional C-extension warnings from torch_geometric
warnings.filterwarnings("ignore", message="An issue occurred while importing")

def run(cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-500:])
    return result.returncode == 0

# ── Detect environment ────────────────────────────────────────────────────────
torch_ver  = torch.__version__.split("+")[0]          # e.g. "2.4.0"
cuda_tag   = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
# e.g. "cu121"
pyg_url    = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"

print(f"PyTorch : {torch_ver}")
print(f"CUDA    : {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"PyG URL : {pyg_url}")

# ── Install torch-geometric (core — always works) ─────────────────────────────
run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])

# ── Try optional C++ extensions (best-effort — warnings are harmless if missing)
for pkg in ["pyg-lib", "torch-scatter", "torch-sparse", "torch-cluster", "torch-spline-conv"]:
    ok = run([sys.executable, "-m", "pip", "install", "-q", pkg, "-f", pyg_url])
    print(f"  {'✓' if ok else '⚠ (optional, skipped)'} {pkg}")

# ── Other dependencies ────────────────────────────────────────────────────────
run([sys.executable, "-m", "pip", "install", "-q",
     "scikit-learn", "matplotlib", "seaborn", "opencv-python",
     "grad-cam", "tqdm", "lime", "shap"])

print("\nAll packages ready.")

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='An issue occurred while importing')

import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, random_split

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import seaborn as sns
import pandas as pd
from PIL import Image

from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

## 2. Device Configuration

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## 3. Dataset Paths

In [ ]:
BASE_PATH = "/kaggle/input/datasets/kostasdiamantaras/chest-xrays-bacterial-viral-pneumonia-normal"

TRAIN_CSV = f"{BASE_PATH}/labels_train.csv"
TRAIN_DIR = f"{BASE_PATH}/train_images/train_images"   # nested one level deeper
TEST_DIR  = f"{BASE_PATH}/test_images/test_images"     # same for test

for p in [TRAIN_CSV, TRAIN_DIR, TEST_DIR]:
    print(f"{'✓' if os.path.exists(p) else '✗'} {p}")

### 3a. Dataset Structure

> Structure confirmed: images live in `train_images/train_images/` and `test_images/test_images/`. Paths are set correctly above.

In [ ]:
# Quick sanity check — count images on disk
import glob
n_train = len(glob.glob(f"{TRAIN_DIR}/*"))
n_test  = len(glob.glob(f"{TEST_DIR}/*"))
print(f"Train images : {n_train}")
print(f"Test  images : {n_test}")

## 4. Inspect Training CSV

In [ ]:
train_csv = pd.read_csv(TRAIN_CSV)
print(train_csv.head())
print("\nColumns:", train_csv.columns.tolist())
print("\nClass distribution:\n", train_csv.iloc[:, 1].value_counts())

## 5. Label Mapping

In [ ]:
CLASSES = ["NORMAL", "BACTERIA", "VIRUS"]
label_map = {c: i for i, c in enumerate(CLASSES)}
print(label_map)

## 6. Data Augmentation & Transforms

> **Fix:** Validation transform no longer includes augmentation — train and val transforms are now separate.

In [ ]:
IMG_SIZE = 224

# Training: augmentation + normalisation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Validation / test: NO augmentation, just resize + normalise
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

## 7. Custom Dataset

> **Fix:** Dataset now auto-resolves filenames using three fallback strategies (flat, per-class subfolders, basename-only walk). A clear `FileNotFoundError` with diagnostic hints replaces the cryptic worker crash.

In [ ]:
class ChestXrayDataset(Dataset):
    """
    Handles both label formats:
      - String labels  : "NORMAL", "BACTERIA", "VIRUS"
      - Integer labels : 0, 1, 2  (already encoded)
    And three file layout variants (flat / nested / per-class subfolders).
    """

    def __init__(self, csv_file, image_dir, transform=None):
        self.data      = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        self._path_cache: dict = {}

        # Build basename → full-path lookup once
        self._disk_index: dict = {}
        for root, _, files in os.walk(image_dir):
            for fname in files:
                self._disk_index[fname] = os.path.join(root, fname)

        # Detect label format from first row
        first_label = self.data.iloc[0, 1]
        if isinstance(first_label, str):
            self._label_fn = lambda x: label_map[x]
        else:
            # Already integer — cast to plain Python int
            self._label_fn = lambda x: int(x)

        print(f"Dataset: {len(self.data)} samples | "
              f"label format: {'string' if isinstance(first_label, str) else 'integer'}")

    def _resolve(self, img_name: str) -> str:
        if img_name in self._path_cache:
            return self._path_cache[img_name]
        candidates = [
            os.path.join(self.image_dir, img_name),
            os.path.join(self.image_dir, os.path.basename(img_name)),
            self._disk_index.get(os.path.basename(img_name), ""),
        ]
        for p in candidates:
            if p and os.path.isfile(p):
                self._path_cache[img_name] = p
                return p
        raise FileNotFoundError(
            f"Image not found: '{img_name}'\n"
            f"  Tried: {[c for c in candidates if c]}"
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_name   = self.data.iloc[idx, 0]
        label_raw  = self.data.iloc[idx, 1]
        img_path   = self._resolve(str(img_name))
        image      = Image.open(img_path).convert("RGB")
        label      = self._label_fn(label_raw)
        if self.transform:
            image = self.transform(image)
        return image, label

## 8. Build Train / Validation Datasets

> **Fix:** `val_dataset` now uses `val_transform` (no augmentation).

In [ ]:
# Build the full dataset TWICE — once per transform — then split by index.
# This ensures val samples are NOT augmented.
full_dataset_train = ChestXrayDataset(TRAIN_CSV, TRAIN_DIR, transform=train_transform)
full_dataset_val   = ChestXrayDataset(TRAIN_CSV, TRAIN_DIR, transform=val_transform)

train_size = int(0.8 * len(full_dataset_train))
val_size   = len(full_dataset_train) - train_size

# Use a shared random seed so both datasets get identical splits
generator = torch.Generator().manual_seed(42)
train_indices, val_indices = random_split(
    range(len(full_dataset_train)), [train_size, val_size], generator=generator
)

from torch.utils.data import Subset
train_dataset = Subset(full_dataset_train, train_indices.indices)
val_dataset   = Subset(full_dataset_val,   val_indices.indices)

print(f"Train samples : {len(train_dataset)}")
print(f"Val   samples : {len(val_dataset)}")

## 9. DataLoaders

> **Improvement:** Added `num_workers` and `pin_memory` for faster GPU throughput.

In [ ]:
BATCH_SIZE  = 32
NUM_WORKERS = 0   # Must be 0 on Kaggle — multiprocessing workers are sandboxed

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Batches — train: {len(train_loader)}, val: {len(val_loader)}")

## 10. Visualise Sample Batch

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

for i in range(8):
    img = images[i].permute(1, 2, 0).numpy()
    img = img * STD + MEAN          # FIX: explicit denormalisation
    img = np.clip(img, 0, 1).astype(np.float32)
    axes[i].imshow(img)
    axes[i].set_title(CLASSES[labels[i].item()], fontsize=11)
    axes[i].axis("off")

plt.suptitle("Sample Training Images", fontsize=14)
plt.tight_layout()
plt.show()

## 11. CNN Backbone — ResNet-50 Feature Extractor

In [ ]:
class ResNet50Extractor(nn.Module):

    def __init__(self):
        super().__init__()
        backbone    = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.features = nn.Sequential(*list(backbone.children())[:-2])
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        feature_map = self.features(x)                       # (B, 2048, 7, 7)
        pooled      = self.pool(feature_map)                 # (B, 2048, 1, 1)
        pooled      = pooled.view(pooled.size(0), -1)        # (B, 2048)
        return feature_map, pooled

## 12. Graph Builder (Spatial Grid)

> **Fix:** `.detach().cpu()` before building graph tensors so gradients don't leak into the graph construction step on GPU.

In [ ]:
def create_graph(feature_map: torch.Tensor):
    """
    Build a 4-connected spatial grid graph from a CNN feature map.
    Each node = one spatial position; node features = channel vector.
    """
    B, C, H, W = feature_map.shape
    graphs = []

    for b in range(B):
        # FIX: detach + move to CPU before numpy/tensor ops
        fmap  = feature_map[b].detach()                        # (C, H, W)
        nodes = fmap.view(C, -1).t().contiguous()              # (H*W, C)

        edge_index = []
        for i in range(H):
            for j in range(W):
                idx = i * W + j
                if j + 1 < W:                                  # right neighbour
                    edge_index += [[idx, idx + 1], [idx + 1, idx]]
                if i + 1 < H:                                  # bottom neighbour
                    edge_index += [[idx, idx + W], [idx + W, idx]]

        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

        graphs.append(Data(x=nodes, edge_index=edge_index))

    return graphs

## 13. Graph Convolutional Network (GCN)

> **Improvement:** Added BatchNorm + Dropout between GCN layers for regularisation.

In [ ]:
class GCN(nn.Module):

    def __init__(self, input_dim: int = 2048, hidden_dim: int = 512, out_dim: int = 128):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.relu  = nn.ReLU()
        self.drop  = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.relu(self.bn1(self.conv1(x, edge_index)))
        x = self.drop(x)
        x = self.relu(self.bn2(self.conv2(x, edge_index)))
        return x

## 14. CNN + GCN Fusion Model

> **Improvement:** Added BatchNorm before classifier; classifier depth increased.

In [ ]:
class FusionModel(nn.Module):

    def __init__(self, num_classes: int = 3):
        super().__init__()
        self.cnn        = ResNet50Extractor()
        self.gcn        = GCN()
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 128, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        feature_map, cnn_features = self.cnn(images)

        graphs      = create_graph(feature_map)
        gcn_outputs = []

        for graph in graphs:
            x          = graph.x.to(DEVICE)
            edge_index = graph.edge_index.to(DEVICE)
            out        = self.gcn(x, edge_index)
            out        = out.mean(dim=0)                  # global mean pooling
            gcn_outputs.append(out)

        gcn_features = torch.stack(gcn_outputs)           # (B, 128)
        fusion       = torch.cat([cnn_features, gcn_features], dim=1)  # (B, 2176)
        return self.classifier(fusion)

## 15. Instantiate Model, Loss, Optimiser & Scheduler

> **Improvement:** Added `ReduceLROnPlateau` scheduler.

In [ ]:
model     = FusionModel(num_classes=3).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)   # label smoothing helps generalisation
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

## 16. Training Loop

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0
    correct = total = 0

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total   += labels.size(0)
        correct += (predicted == labels).sum().item()

    return running_loss / len(loader), correct / total

## 17. Validation Loop

In [ ]:
def validate(model, loader):
    model.eval()
    running_loss = 0.0
    correct = total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="  Val  ", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss / len(loader), correct / total

## 18. Train the Model

In [ ]:
EPOCHS       = 15           # increased from 10
best_val_acc = 0.0

train_losses, val_losses         = [], []
train_accuracies, val_accuracies = [], []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}  (lr={optimizer.param_groups[0]['lr']:.2e})")

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss,   val_acc   = validate(model, val_loader)

    scheduler.step(val_acc)

    train_losses.append(train_loss);     val_losses.append(val_loss)
    train_accuracies.append(train_acc);  val_accuracies.append(val_acc)

    print(f"  Train → loss: {train_loss:.4f}  acc: {train_acc:.4f}")
    print(f"  Val   → loss: {val_loss:.4f}  acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_fusion_model.pth")
        print("  ✓ Best model saved!")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

## 19. Load Best Model

> **Fix:** Added `map_location=DEVICE` and `weights_only=True` to silence FutureWarning and correctly load on any device.

In [ ]:
model.load_state_dict(
    torch.load("best_fusion_model.pth", map_location=DEVICE, weights_only=True)
)
model.eval()
print("Best model loaded successfully.")

## 20. Collect Validation Predictions

In [ ]:
all_preds  = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images  = images.to(DEVICE, non_blocking=True)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print("Evaluation complete.")

## 21. Classification Report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASSES))

## 22. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted");  plt.ylabel("True")
plt.title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

## 23. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_accuracies, label="Train Acc", marker="o")
axes[0].plot(val_accuracies,   label="Val Acc",   marker="s")
axes[0].set_title("Accuracy");  axes[0].set_xlabel("Epoch");  axes[0].set_ylabel("Accuracy")
axes[0].legend();  axes[0].grid(True, alpha=0.3)

axes[1].plot(train_losses, label="Train Loss", marker="o")
axes[1].plot(val_losses,   label="Val Loss",   marker="s")
axes[1].set_title("Loss");  axes[1].set_xlabel("Epoch");  axes[1].set_ylabel("Loss")
axes[1].legend();  axes[1].grid(True, alpha=0.3)

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
plt.show()

## 27. Explainable AI (XAI) Suite

Five complementary explanation techniques, each answering a different question:

| Technique | Question answered |
|---|---|
| **Grad-CAM** | *Which spatial regions drove the prediction?* |
| **Grad-CAM++** | *Better localisation for multi-instance cases* |
| **LIME** | *Which superpixels pushed the class probability up/down?* |
| **SHAP (DeepSHAP)** | *Per-pixel attribution relative to a background distribution* |
| **Occlusion Sensitivity** | *How does prediction change as we slide a grey patch across the image?* |
| **Attention Rollout (CNN)** | *Where does the model attend across all conv layers?* |
| **Per-class probability bars** | *How confident is the model, per class?* |


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","lime","shap"])
print("LIME & SHAP installed.")

In [ ]:
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

MEAN_NP = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD_NP  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def denorm(tensor_img):
    """CHW tensor → HWC float32 numpy in [0,1]."""
    img = tensor_img.permute(1,2,0).numpy().astype(np.float32)
    img = img * STD_NP + MEAN_NP
    return np.clip(img, 0, 1)

def get_sample(dataset_idx):
    """Return (tensor, rgb_numpy, true_label_int) for one val sample."""
    img_t, label = val_dataset[dataset_idx]
    return img_t, denorm(img_t), int(label)

print("XAI helpers ready.")

In [ ]:
# ── Shared CAM wrapper (CNN branch only — needed for all CAM methods) ────────
class GradCAMWrapper(nn.Module):
    def __init__(self, fusion_model):
        super().__init__()
        self.cnn        = fusion_model.cnn
        self.classifier = fusion_model.classifier
        self._gcn_zeros = torch.zeros(1, 128, device=DEVICE)

    def forward(self, x):
        _, cnn_feat = self.cnn(x)
        gcn_dummy   = self._gcn_zeros.expand(x.size(0), -1)
        return self.classifier(torch.cat([cnn_feat, gcn_dummy], dim=1))

cam_model     = GradCAMWrapper(model).to(DEVICE).eval()
target_layers = [cam_model.cnn.features[-1]]
print("CAM wrapper ready.")

### 27.1 Grad-CAM vs Grad-CAM++

Grad-CAM weights feature maps by their gradient signal. Grad-CAM++ uses second-order gradients to better handle cases where multiple instances of the same class appear in one image.

In [ ]:
def plot_cam_comparison(idx=0):
    img_t, rgb, true_lbl = get_sample(idx)
    inp = img_t.unsqueeze(0).to(DEVICE)
    target = [ClassifierOutputTarget(true_lbl)]

    with GradCAM(model=cam_model, target_layers=target_layers) as gc:
        cam1 = gc(input_tensor=inp, targets=target)[0]

    with GradCAMPlusPlus(model=cam_model, target_layers=target_layers) as gcpp:
        cam2 = gcpp(input_tensor=inp, targets=target)[0]

    with torch.no_grad():
        probs = torch.softmax(model(inp), dim=1)[0].cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb);                              axes[0].set_title(f"Original\nTrue: {CLASSES[true_lbl]}")
    axes[1].imshow(show_cam_on_image(rgb, cam1));     axes[1].set_title("Grad-CAM")
    axes[2].imshow(show_cam_on_image(rgb, cam2));     axes[2].set_title("Grad-CAM++")
    for ax in axes: ax.axis("off")
    plt.suptitle(f"Predicted: {CLASSES[probs.argmax()]}  "
                 f"(conf: {probs.max():.1%})", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

    # Confidence bar chart
    fig, ax = plt.subplots(figsize=(6, 3))
    bars = ax.barh(CLASSES, probs, color=["#4CAF50" if i==true_lbl else "#2196F3" for i in range(3)])
    ax.bar_label(bars, fmt="%.3f", padding=4)
    ax.set_xlim(0, 1.1); ax.set_xlabel("Probability")
    ax.set_title("Class Confidence")
    plt.tight_layout(); plt.show()

plot_cam_comparison(0)
plot_cam_comparison(10)

### 27.2 Class-Specific Activation Maps

Generate a Grad-CAM map *for each class* (not just the predicted one). This reveals what the model would focus on if it were to predict each class — useful for understanding decision boundaries.

In [ ]:
def plot_all_class_cams(idx=0):
    img_t, rgb, true_lbl = get_sample(idx)
    inp = img_t.unsqueeze(0).to(DEVICE)

    fig, axes = plt.subplots(1, len(CLASSES)+1, figsize=(5*(len(CLASSES)+1), 5))
    axes[0].imshow(rgb); axes[0].set_title(f"Original\nTrue: {CLASSES[true_lbl]}"); axes[0].axis("off")

    with GradCAMPlusPlus(model=cam_model, target_layers=target_layers) as gcpp:
        for ci, cls_name in enumerate(CLASSES):
            cam_map = gcpp(input_tensor=inp, targets=[ClassifierOutputTarget(ci)])[0]
            axes[ci+1].imshow(show_cam_on_image(rgb, cam_map))
            axes[ci+1].set_title(f"If → {cls_name}")
            axes[ci+1].axis("off")

    plt.suptitle("Class-Specific Activation Maps", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

plot_all_class_cams(0)

### 27.3 Occlusion Sensitivity

A grey patch is slid across the image. The heatmap shows how much the predicted-class probability *drops* when each region is occluded — bright = that region is critical to the prediction.

In [ ]:
def occlusion_sensitivity(idx=0, patch_size=32, stride=16):
    img_t, rgb, true_lbl = get_sample(idx)
    inp = img_t.unsqueeze(0).to(DEVICE)

    # Baseline probability for the true class
    model.eval()
    with torch.no_grad():
        base_prob = torch.softmax(model(inp), dim=1)[0, true_lbl].item()

    C, H, W = img_t.shape
    heatmap  = np.zeros((H, W), dtype=np.float32)
    counts   = np.zeros((H, W), dtype=np.float32)

    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            occluded = inp.clone()
            occluded[:, :, y:y+patch_size, x:x+patch_size] = 0.5  # grey patch
            with torch.no_grad():
                prob = torch.softmax(model(occluded), dim=1)[0, true_lbl].item()
            drop = base_prob - prob          # positive = region was important
            heatmap[y:y+patch_size, x:x+patch_size] += drop
            counts[y:y+patch_size,  x:x+patch_size] += 1

    counts = np.maximum(counts, 1)
    heatmap = heatmap / counts
    heatmap = np.clip(heatmap, 0, None)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(rgb);                                           axes[0].set_title("Original")
    im = axes[1].imshow(heatmap, cmap="hot");                      axes[1].set_title("Occlusion Sensitivity")
    plt.colorbar(im, ax=axes[1])
    blend = (rgb * 0.5 + plt.cm.hot(heatmap)[...,:3] * 0.5).clip(0,1)
    axes[2].imshow(blend);                                         axes[2].set_title("Overlay")
    for ax in axes: ax.axis("off")
    plt.suptitle(f"Occlusion Sensitivity — True: {CLASSES[true_lbl]}", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

occlusion_sensitivity(0)
occlusion_sensitivity(5)

### 27.4 LIME (Local Interpretable Model-agnostic Explanations)

LIME segments the image into superpixels and perturbs them to fit a local linear model. Green superpixels *support* the predicted class; red superpixels *contradict* it.

In [ ]:
def lime_explain(idx=0, num_samples=500):
    img_t, rgb, true_lbl = get_sample(idx)

    def predict_fn(images_np):
        """LIME passes HWC uint8 numpy arrays; return class probabilities."""
        preds = []
        for img in images_np:
            pil = Image.fromarray((img).astype(np.uint8))
            t   = val_transform(pil).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                prob = torch.softmax(model(t), dim=1)[0].cpu().numpy()
            preds.append(prob)
        return np.array(preds)

    explainer    = lime_image.LimeImageExplainer()
    rgb_uint8    = (rgb * 255).astype(np.uint8)
    explanation  = explainer.explain_instance(
        rgb_uint8, predict_fn,
        top_labels=3, hide_color=0, num_samples=num_samples
    )

    pred_class = explanation.top_labels[0]
    temp, mask = explanation.get_image_and_mask(
        pred_class, positive_only=False, num_features=8, hide_rest=False
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(rgb);                                          axes[0].set_title(f"Original\nTrue: {CLASSES[true_lbl]}")
    axes[1].imshow(mark_boundaries(temp / 255.0, mask));          axes[1].set_title(f"LIME — Pred: {CLASSES[pred_class]}\n(green=supports, red=contradicts)")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

lime_explain(0)

### 27.5 SHAP DeepExplainer

DeepSHAP uses a background reference set and backpropagation to assign each pixel a Shapley value — the average marginal contribution to the prediction. Red = increases predicted class score; blue = decreases it.

In [ ]:
def shap_explain(idx=0, n_background=16, n_samples=100):
    """
    SHAP GradientExplainer — compatible with ResNet skip connections.
    GradientExplainer.shap_values() returns either:
      - a list of arrays (one per class)  → shape each: (1, 3, H, W)
      - a single array                    → shape: (1, num_classes, 3, H, W) or (num_classes, 1, 3, H, W)
    We normalise both cases below.
    """
    img_t, rgb, true_lbl = get_sample(idx)
    inp = img_t.unsqueeze(0).to(DEVICE)

    # Background
    rng_idx    = np.random.choice(len(val_dataset), n_background, replace=False)
    background = torch.stack([val_dataset[bi][0] for bi in rng_idx]).to(DEVICE)

    explainer   = shap.GradientExplainer(cam_model, background)
    raw         = explainer.shap_values(inp, nsamples=n_samples)

    # ── Normalise output to list of (3, H, W) arrays, one per class ──────────
    if isinstance(raw, list):
        # list of (1, 3, H, W)  →  list of (3, H, W)
        per_class = [r[0] for r in raw]
    else:
        raw = np.array(raw)
        if raw.ndim == 5:
            # (num_classes, 1, 3, H, W)  or  (1, num_classes, 3, H, W)
            if raw.shape[0] == len(CLASSES):
                per_class = [raw[ci, 0] for ci in range(len(CLASSES))]
            else:
                per_class = [raw[0, ci] for ci in range(len(CLASSES))]
        elif raw.ndim == 4:
            # (1, 3, H, W) — only one output; duplicate across classes for display
            per_class = [raw[0]] * len(CLASSES)
        else:
            raise ValueError(f"Unexpected shap_values shape: {raw.shape}")

    print(f"SHAP per-class array shapes: {[p.shape for p in per_class]}")

    # Average over RGB channels → (H, W) per class
    shap_maps = [p.transpose(1, 2, 0).mean(axis=-1) for p in per_class]
    vmax      = max(np.abs(m).max() for m in shap_maps)

    fig, axes = plt.subplots(1, len(CLASSES) + 1, figsize=(5 * (len(CLASSES) + 1), 5))
    axes[0].imshow(rgb)
    axes[0].set_title(f"Original\nTrue: {CLASSES[true_lbl]}")
    axes[0].axis("off")

    for ci, cls_name in enumerate(CLASSES):
        im = axes[ci + 1].imshow(shap_maps[ci], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axes[ci + 1].set_title(f"SHAP → {cls_name}")
        axes[ci + 1].axis("off")
        plt.colorbar(im, ax=axes[ci + 1], fraction=0.046)

    plt.suptitle("SHAP GradientExplainer — Per-class Attribution\n"
                 "(red = increases class score, blue = decreases it)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

shap_explain(0)

### 27.6 Full XAI Dashboard (all methods, one sample)

Runs all five methods on a single sample and displays them in one figure for easy side-by-side comparison.

In [ ]:
def xai_dashboard(idx=0):
    img_t, rgb, true_lbl = get_sample(idx)
    inp = img_t.unsqueeze(0).to(DEVICE)

    # ── Probabilities ──────────────────────────────────────────────────────
    with torch.no_grad():
        probs = torch.softmax(model(inp), dim=1)[0].cpu().numpy()
    pred_lbl = int(probs.argmax())

    # ── Grad-CAM++ ────────────────────────────────────────────────────────
    with GradCAMPlusPlus(model=cam_model, target_layers=target_layers) as gcpp:
        cam_map = gcpp(inp, targets=[ClassifierOutputTarget(pred_lbl)])[0]
    cam_overlay = show_cam_on_image(rgb, cam_map)

    # ── Occlusion ─────────────────────────────────────────────────────────
    base_p = probs[pred_lbl]; patch, stride = 28, 14
    H = W = 224
    hmap = np.zeros((H,W)); cnt = np.zeros((H,W))
    for y in range(0, H-patch+1, stride):
        for x in range(0, W-patch+1, stride):
            occ = inp.clone()
            occ[:,:,y:y+patch,x:x+patch] = 0.5
            with torch.no_grad():
                p = torch.softmax(model(occ),dim=1)[0,pred_lbl].item()
            hmap[y:y+patch,x:x+patch] += base_p - p
            cnt[y:y+patch, x:x+patch] += 1
    hmap /= np.maximum(cnt,1)
    hmap  = np.clip(hmap,0,None)
    if hmap.max()>0: hmap /= hmap.max()
    occ_overlay = (rgb*0.5 + plt.cm.hot(hmap)[...,:3]*0.5).clip(0,1)

    # ── Layout ────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(20, 10))
    gs  = fig.add_gridspec(2, 4, hspace=0.35, wspace=0.25)

    ax_orig  = fig.add_subplot(gs[0, 0])
    ax_cam   = fig.add_subplot(gs[0, 1])
    ax_occ   = fig.add_subplot(gs[0, 2])
    ax_conf  = fig.add_subplot(gs[0, 3])
    ax_cls0  = fig.add_subplot(gs[1, 0])
    ax_cls1  = fig.add_subplot(gs[1, 1])
    ax_cls2  = fig.add_subplot(gs[1, 2])
    ax_note  = fig.add_subplot(gs[1, 3])

    ax_orig.imshow(rgb);         ax_orig.set_title("Original Image");           ax_orig.axis("off")
    ax_cam.imshow(cam_overlay);  ax_cam.set_title("Grad-CAM++ (predicted cls)"); ax_cam.axis("off")
    ax_occ.imshow(occ_overlay);  ax_occ.set_title("Occlusion Sensitivity");      ax_occ.axis("off")

    colors = ["#4CAF50" if i==true_lbl else ("#F44336" if i==pred_lbl else "#90CAF9") for i in range(3)]
    bars = ax_conf.barh(CLASSES, probs, color=colors)
    ax_conf.bar_label(bars, fmt="%.3f", padding=3)
    ax_conf.set_xlim(0,1.15); ax_conf.set_title("Class Probabilities")
    ax_conf.set_xlabel("Softmax probability")

    # Class-specific CAMs (bottom row)
    with GradCAMPlusPlus(model=cam_model, target_layers=target_layers) as gcpp:
        for ci, (ax, cls_name) in enumerate(zip([ax_cls0,ax_cls1,ax_cls2], CLASSES)):
            cm2 = gcpp(inp, targets=[ClassifierOutputTarget(ci)])[0]
            ax.imshow(show_cam_on_image(rgb, cm2))
            ax.set_title(f"CAM if → {cls_name}"); ax.axis("off")

    # Legend
    ax_note.axis("off")
    legend_txt = (
        "Colour key (probabilities):\n"
        "  🟩 Green  = true class\n"
        "  🟥 Red    = predicted class (if wrong)\n"
        "  🟦 Blue   = other class\n\n"
        "Occlusion heatmap:\n"
        "  Bright = critical region\n\n"
        "Bottom row = what the model\n"
        "would attend to if forced\n"
        "to predict each class."
    )
    ax_note.text(0.05, 0.95, legend_txt, transform=ax_note.transAxes,
                 va="top", fontsize=10, family="monospace",
                 bbox=dict(boxstyle="round", fc="#f5f5f5", ec="#bdbdbd"))

    correct = "✓ Correct" if pred_lbl==true_lbl else "✗ Wrong"
    fig.suptitle(
        f"XAI Dashboard  |  True: {CLASSES[true_lbl]}  |  "
        f"Predicted: {CLASSES[pred_lbl]} ({probs[pred_lbl]:.1%})  |  {correct}",
        fontsize=14, fontweight="bold"
    )
    plt.show()

xai_dashboard(0)
xai_dashboard(5)
xai_dashboard(15)

## 24. Grad-CAM Visualisation

> **Fix:** Added a `GradCAMWrapper` that strips the GCN forward path so GradCAM can cleanly hook into the CNN backbone without the graph ops interfering with gradient flow.

In [ ]:
class GradCAMWrapper(nn.Module):
    """Expose only the CNN + classifier path for GradCAM compatibility."""
    def __init__(self, fusion_model: FusionModel):
        super().__init__()
        self.cnn        = fusion_model.cnn
        self.classifier = fusion_model.classifier
        # Fixed zero GCN output so shapes still match
        self._gcn_zeros = torch.zeros(1, 128, device=DEVICE)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, cnn_features = self.cnn(x)                           # (B, 2048)
        gcn_dummy       = self._gcn_zeros.expand(x.size(0), -1)
        fusion          = torch.cat([cnn_features, gcn_dummy], dim=1)
        return self.classifier(fusion)


cam_wrapper   = GradCAMWrapper(model).to(DEVICE).eval()
target_layers = [cam_wrapper.cnn.features[-1]]                  # last ResNet conv block

cam = GradCAM(model=cam_wrapper, target_layers=target_layers)

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def show_gradcam(dataset_idx: int = 0):
    image, label = val_dataset[dataset_idx]
    input_tensor = image.unsqueeze(0).to(DEVICE)

    rgb_img = image.permute(1, 2, 0).numpy()
    rgb_img = (rgb_img * STD + MEAN).clip(0, 1).astype(np.float32)

    grayscale_cam = cam(input_tensor=input_tensor)[0]           # FIX: removed stray ')'
    cam_image     = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(rgb_img);      axes[0].set_title(f"Original — True: {CLASSES[label]}")
    axes[1].imshow(cam_image);    axes[1].set_title("Grad-CAM Overlay")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()

show_gradcam(0)
show_gradcam(5)

## 25. Single-Image Prediction

In [ ]:
def predict_and_show(dataset_idx: int = 5):
    image, label = val_dataset[dataset_idx]
    input_tensor = image.unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]
        _, pred = torch.max(outputs, 1)

    predicted_class = CLASSES[pred.item()]
    true_class      = CLASSES[label]

    print(f"True Label : {true_class}")
    print(f"Predicted  : {predicted_class}")
    print("\nClass probabilities:")
    for c, p in zip(CLASSES, probs):
        print(f"  {c:<10}: {p:.4f}")

    img = image.permute(1, 2, 0).numpy()
    img = (img * STD + MEAN).clip(0, 1).astype(np.float32)

    plt.figure(figsize=(5, 5))
    plt.imshow(img)
    colour = "green" if predicted_class == true_class else "red"
    plt.title(f"Pred: {predicted_class}  |  True: {true_class}", color=colour, fontsize=13)
    plt.axis("off")
    plt.show()

predict_and_show(5)

## 26. Summary

In [ ]:
print("=" * 50)
print("  EXPERIMENT SUMMARY")
print("=" * 50)
print(f"  Best Validation Accuracy : {best_val_acc:.4f}")
print(f"  Final Val Accuracy       : {val_accuracies[-1]:.4f}")
print(f"  Final Val Loss           : {val_losses[-1]:.4f}")
print(f"  Epochs trained           : {EPOCHS}")
print("=" * 50)